In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6bde1c1d10c045c3b38ed91e87143baa6d785eee0600cfe1ff667255583f5e38
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.
#
# BB84 (Bennett & Brassard, 1984) is a quantum key distribution protocol.
# Alice and Bob communicate over a public quantum channel to establish a
# shared secret key. Because quantum states collapse on measurement,
# any eavesdropping is detectable.
#
# Basis convention used throughout:
#   basis = 0  ->  standard  basis: |0> represents bit 0, |1> represents bit 1
#   basis = 1  ->  diagonal  basis: |+> represents bit 0, |-> represents bit 1
#
# All random choices are generated by measuring the quantum state
# |+> = (1/sqrt(2))(|0> + |1>), which gives 0 or 1 with equal probability.
# This is genuine quantum randomness, not a pseudo-random number generator.

In [2]:
# ============================================================
#  SETUP: Simulator and helper functions
# ============================================================

simulator = BasicSimulator()

# Number of qubits Alice will send.
# After sifting (~half discarded) and sacrificing a sample for checking,
# the final key will be roughly (N/2 - SAMPLE_SIZE) bits.
N = 100
SAMPLE_SIZE = 20   # bits sacrificed for eavesdrop detection check
DETECTION_THRESHOLD = 0.10   # flag attack if error rate > 10%


def random_bit():
    """
    Generate a single genuinely random bit using quantum measurement.

    Constructs the state |+> = H|0> = (1/sqrt(2))(|0> + |1>) and measures it.
    Each outcome (0 or 1) occurs with probability 1/2 by the Born rule.
    This is true randomness -- not pseudo-random.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # apply Hadamard: |0> -> |+>
    qc.measure(0, 0) # collapse |+> to |0> or |1> with equal probability
    compiled = transpile(qc, simulator)
    result   = simulator.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])


def encode_qubit(bit, basis):
    """
    Encode a classical bit as a qubit in the chosen basis.

    Standard basis (basis=0):
        bit 0  ->  |0>        (do nothing; qubits initialise to |0>)
        bit 1  ->  |1>        (apply X to flip)

    Diagonal basis (basis=1):
        bit 0  ->  |+> = H|0>        (apply H)
        bit 1  ->  |-> = H|1> = HX|0>  (apply X then H)

    Returns a QuantumCircuit with the qubit prepared but not yet measured.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)       # flip to |1> if bit is 1
    if basis == 1:    # diagonal basis: apply H to convert |0>->|+>, |1>->|->
        qc.h(0)
    return qc


def measure_qubit(qc, basis):
    """
    Measure a qubit (carried in QuantumCircuit qc) in the given basis.

    Standard basis (basis=0):  measure directly in the {|0>,|1>} basis.
    Diagonal basis (basis=1):  apply H first, then measure.
        H converts |+> -> |0>  and  |-> -> |1>, so the standard
        measurement then reliably reads the diagonal-basis state.

    Returns the measured classical bit (0 or 1).
    """
    if basis == 1:    # diagonal measurement: rotate back with H
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result   = simulator.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])


def basis_name(b):
    """Return a readable label for basis index b."""
    return 'standard' if b == 0 else 'diagonal'

In [3]:
# ============================================================
#  ALICE
#  Step 1: Generate a random bit string and random basis choices.
#  Step 2: Encode each bit as a qubit and "send" it to Bob.
# ============================================================

print('=== ALICE ===')
print(f'Preparing {N} qubits to send to Bob.')
print('All random choices are made by measuring |+> (quantum randomness).\n')

alice_bits  = [random_bit() for _ in range(N)]  # secret bit string
alice_bases = [random_bit() for _ in range(N)]  # 0=standard, 1=diagonal

# Encode each bit as a qubit circuit.
# In a real QKD system these would be photons sent over a fibre;
# here we store the circuit objects and pass them to Bob directly.
transmitted_qubits = []
for i in range(N):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    transmitted_qubits.append(qc)

print(f'Alice bits  (first 20): {alice_bits[:20]}')
print(f'Alice bases (first 20): {[basis_name(b) for b in alice_bases[:20]]}')
print('\nAlice has encoded her bits as qubits and sent them to Bob.')
print('Alice keeps her bit string and basis choices PRIVATE for now.')

=== ALICE ===
Preparing 100 qubits to send to Bob.
All random choices are made by measuring |+> (quantum randomness).

Alice bits  (first 20): [0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1]
Alice bases (first 20): ['diagonal', 'standard', 'standard', 'standard', 'diagonal', 'diagonal', 'diagonal', 'diagonal', 'standard', 'diagonal', 'standard', 'standard', 'diagonal', 'standard', 'diagonal', 'diagonal', 'diagonal', 'diagonal', 'diagonal', 'diagonal']

Alice has encoded her bits as qubits and sent them to Bob.
Alice keeps her bit string and basis choices PRIVATE for now.


In [4]:
# ============================================================
#  BOB
#  Step 3: For each received qubit, independently choose a random
#           measurement basis and record the outcome.
# ============================================================

print('=== BOB ===')
print('Bob receives the qubits and measures each one in a randomly chosen basis.\n')

bob_bases   = [random_bit() for _ in range(N)]   # Bob's independent random bases
bob_results = []

for i in range(N):
    # Each transmitted qubit is a QuantumCircuit object.
    # We compose it with Bob's measurement operation.
    qc = QuantumCircuit(1, 1)
    qc.compose(transmitted_qubits[i], inplace=True)   # apply Alice's encoding
    bit = measure_qubit(qc, bob_bases[i])             # Bob measures
    bob_results.append(bit)

print(f'Bob bases   (first 20): {[basis_name(b) for b in bob_bases[:20]]}')
print(f'Bob results (first 20): {bob_results[:20]}')
print('\nBob has measured all qubits. His results are private for now.')

=== BOB ===
Bob receives the qubits and measures each one in a randomly chosen basis.

Bob bases   (first 20): ['diagonal', 'diagonal', 'diagonal', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'standard', 'diagonal', 'standard', 'standard', 'standard']
Bob results (first 20): [0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0]

Bob has measured all qubits. His results are private for now.


In [5]:
# ============================================================
#  PUBLIC CHANNEL: BASIS RECONCILIATION (SIFTING)
#  Alice and Bob publicly compare their basis choices.
#  They keep only the positions where both used the same basis.
#  The actual bit values remain secret.
# ============================================================

print('=== PUBLIC CHANNEL: BASIS RECONCILIATION ===')
print('Bob announces his basis choices. Alice replies with which ones match.\n')

# Find positions where Alice and Bob chose the same basis.
# When bases match, Bob's measurement result equals Alice's original bit (no noise/attacker).
# When bases differ, Bob's result is random -- those positions are discarded.
matching_positions = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]   for i in matching_positions]
bob_sifted   = [bob_results[i]  for i in matching_positions]

print(f'Total qubits sent:       {N}')
print(f'Matching basis positions: {len(matching_positions)} '
      f'({100*len(matching_positions)/N:.1f}% -- expected ~50%)')
print(f'\nSifted key length: {len(alice_sifted)} bits')
print(f'Alice sifted (first 20): {alice_sifted[:20]}')
print(f'Bob   sifted (first 20): {bob_sifted[:20]}')

=== PUBLIC CHANNEL: BASIS RECONCILIATION ===
Bob announces his basis choices. Alice replies with which ones match.

Total qubits sent:       100
Matching basis positions: 49 (49.0% -- expected ~50%)

Sifted key length: 49 bits
Alice sifted (first 20): [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0]
Bob   sifted (first 20): [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0]


In [6]:
# ============================================================
#  PUBLIC CHANNEL: EAVESDROP DETECTION CHECK
#  Alice and Bob publicly compare a RANDOM sample of their sifted bits.
#  Sample indices are chosen using quantum randomness (random_bit()).
#  If the error rate exceeds a threshold, an attack is declared.
#  The sacrificed sample bits are discarded from the final key.
# ============================================================

print('=== PUBLIC CHANNEL: EAVESDROP DETECTION CHECK ===')
print(f'Alice and Bob publicly compare a random sample of {SAMPLE_SIZE} sifted bits.\n')

# Select SAMPLE_SIZE unique indices using quantum randomness (rejection sampling).
seen = set()
sample_indices = []
while len(sample_indices) < SAMPLE_SIZE:
    bits_needed = max(1, (len(alice_sifted) - 1).bit_length())
    idx = sum(random_bit() << k for k in range(bits_needed)) % len(alice_sifted)
    if idx not in seen:
        seen.add(idx)
        sample_indices.append(idx)
sample_indices.sort()

sample_alice = [alice_sifted[i] for i in sample_indices]
sample_bob   = [bob_sifted[i]   for i in sample_indices]

errors     = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / SAMPLE_SIZE

print(f'Sample (Alice): {sample_alice}')
print(f'Sample (Bob):   {sample_bob}')
print(f'Errors in sample: {errors} / {SAMPLE_SIZE}')
print(f'Error rate: {error_rate:.2%}')
print(f'Detection threshold: {DETECTION_THRESHOLD:.0%}\n')

if error_rate > DETECTION_THRESHOLD:
    print('*** ATTACK DETECTED! Error rate exceeds threshold. ***')
    print('Alice and Bob abort the key exchange.')
else:
    print('No attack detected. Error rate is within the expected range.')

    # Discard the sample bits; the rest form the final key
    key_indices     = [i for i in range(len(alice_sifted)) if i not in set(sample_indices)]
    final_key_alice = [alice_sifted[i] for i in key_indices]
    final_key_bob   = [bob_sifted[i]   for i in key_indices]

    print(f'\nFinal shared key length: {len(final_key_alice)} bits')
    print(f'Alice key (first 20 bits): {final_key_alice[:20]}')
    print(f'Bob   key (first 20 bits): {final_key_bob[:20]}')

    keys_agree = (final_key_alice == final_key_bob)
    print(f'\nKeys identical: {keys_agree}')
    if keys_agree:
        print('SUCCESS: Alice and Bob share the same secret key!')
        print('This key can now be used as a one-time pad for perfectly secure communication.')

=== PUBLIC CHANNEL: EAVESDROP DETECTION CHECK ===
Alice and Bob publicly compare a random sample of 20 sifted bits.

Sample (Alice): [0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0]
Sample (Bob):   [0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0]
Errors in sample: 0 / 20
Error rate: 0.00%
Detection threshold: 10%

No attack detected. Error rate is within the expected range.

Final shared key length: 29 bits
Alice key (first 20 bits): [0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0]
Bob   key (first 20 bits): [0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0]

Keys identical: True
SUCCESS: Alice and Bob share the same secret key!
This key can now be used as a one-time pad for perfectly secure communication.
